# Spicerack - Recipe Recommender
### **DS Club Project - Spring 2026**


**SpiceRack** is a recipe recommendation system that takes the spices in your pantry and finds recipes you can make. It learns which spices are rare and distinctive (saffron, galangal) vs common and uninformative (salt, pepper); the results are driven by what makes your pantry unique.


### - **Features**
* Recommends recipes based on your spice pantry using a trained K-Means model
* Hearts recipes to save them to your personal collection
* Suggests spices to buy that would unlock the most new recipes
* Scans barcodes on spice jars to add spices automatically
* Validates spice input  only accepts known canonical spices from a 179-spice vocabulary
* Modal popups with full ingredients, directions, and a photo from Unsplash


###  - **Project Resources**
* Main Repository: [GitHub - SpiceRack Core](https://github.com/laniel123/DS-Club-Project-Showcase-SpiceRack)
* Web Implementation: [GitHub - SpiceRack UI](https://github.com/Luke-B-Mal/SpiceRack-website)
* Data Source: [Kaggle Data](https://www.kaggle.com/datasets/paultimothymooney/recipenlg/data)
* Restrictions Logic: [GitHub - Recipes-Restrictions](https://github.com/austinpak/Recipes-Restrictions)
* Image API: [Unsplash](https://unsplash.com/)
* Barcode API: [Open Food Facts](https://world.openfoodfacts.org/)



---
## - Change your inputs here

This is the only cell you need to touch. Edit your pantry, then run all cells. use for the initial testing of the model within this notebook.

If you put something in must-use that isn't in your pantry, it gets added automatically.
If a spice isn't recognized it just gets skipped with a warning.

In [42]:
# change these two lists and run everything

# spices you actually have right now
MY_PANTRY = [
    "garlic",
    "cumin",
    "paprika",
    "chili powder",
    "oregano",
    "black pepper",
    "salt",
    "cinnamon",
    "ginger",
]

# spices that MUST show up in every recipe we recommend
# leave as [] if you dont care
MUST_USE = [
    "cumin",
    "garlic",
]

# path to your original csv file downloaded on kaggle 
CSV_PATH = "cookingdataset/RecipeNLG_dataset.csv"

# how many recipes to load - lower this if its running slow
SAMPLE_SIZE = None  # set to None to load all, or a number like 10000 for a sample

---
## - Imports and setup
Run this once, dont touch it

In [ ]:
import re
import warnings
import pandas as pd
import numpy as np
import joblib
import time
from sklearn.decomposition import NMF
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import pairwise_distances

# pulling all our spice data from the external file
# spice_data.py needs to be in the same folder as this notebook
from spice_data_v2 import SPICES, ALIASES, CANONICAL_SPICES, FLAVOR_PROFILES, REGION_PROFILES

print(f"loaded {len(CANONICAL_SPICES)} spices, {len(FLAVOR_PROFILES)} flavor profiles, {len(REGION_PROFILES)} regions")

loaded 179 spices, 15 flavor profiles, 16 regions


In [2]:
# helper functions used throughout

from spice_data_v2 import SPICES, ALIASES, CANONICAL_SPICES

def clean(s) -> str:
    # lowercase and strip weird characters
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"[^a-z\s']", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def to_canonical(spice):
    # normalize a spice name and apply alias mapping
    n = clean(spice)
    return clean(ALIASES.get(n, n))


def validate_spices(raw_list, label):
    # make sure everything the user typed is actually in our vocabulary
    # warns on anything we dont recognize and just skips it
    valid = set()
    bad = []
    for s in raw_list:
        c = to_canonical(s)
        if c in CANONICAL_SPICES:
            valid.add(c)
        else:
            bad.append(s)
    if bad:
        print(f"warning ({label}): didnt recognize these, skipping: {bad}")
    return valid


# build regex patterns for spice extraction from recipe text
# sorted by length so longer matches beat shorter ones (smoked paprika before paprika)
SPICE_PATTERNS = sorted(
    [(sp, clean(sp), re.compile(rf"(^| ){re.escape(clean(sp))}( |$)")) for sp in SPICES],
    key=lambda x: -len(x[0])
)

def get_spices_from_recipe(ingredients):
    # takes a recipe's ingredient list and returns which spices are in it
    if isinstance(ingredients, list):
        raw = " ".join(str(x) for x in ingredients)
    else:
        raw = str(ingredients)
    text = clean(raw)
    found = set()
    for _, norm, pat in SPICE_PATTERNS:
        if pat.search(" " + text + " "):
            found.add(norm)
    return {to_canonical(sp) for sp in found}


def parse_ingredient_string(x) -> list[str]:
    # ingredients are stored as a stringified python list in the csv
    # like '["1 cup flour", "2 eggs"]' so we parse it back
    if isinstance(x, list):
        return [str(i) for i in x]
    if not isinstance(x, str):
        return []
    s = x.strip()
    if s.startswith("[") and s.endswith("]"):
        items = re.findall(r"'([^']*)'|\"([^\"]*)\"", s)
        parsed = [a if a else b for a, b in items]
        return parsed if parsed else [s]
    return [s]



print("helper functions ready")

helper functions ready


---
## - Load the Kaggle data and create the model

* [Kaggle Data](https://www.kaggle.com/datasets/paultimothymooney/recipenlg/data) 
* Run this once per session. Takes a minute on the full dataset.

In [8]:
#df = pd.read_csv(CSV_PATH)

In [9]:
#df.columns

In [10]:
#df.head()

In [ ]:
#sample_df = df.sample(n=1000, random_state=42)

#sample_df = df.sample(n=1000, random_state=42)

# save a smaller sample for quick testing without loading the whole thing every time

#sample_df.columns
#sample_df.to_csv('sample_dataset_1000.csv', index=False)

In [ ]:
#cooking data
df = pd.read_csv(CSV_PATH)

In [ ]:
print(f"loading {CSV_PATH}...")

if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# parse NER column (clean ingredient tokens) instead of raw ingredients
# NER gives ["vanilla", "flour", "butter"] not "1/2 tsp. vanilla extract"
df["ingredients_parsed"] = df["NER"].apply(parse_ingredient_string)
df["spices"] = df["ingredients_parsed"].apply(get_spices_from_recipe)

# binary matrix: rows = recipes, columns = spices, 1 if recipe contains that spice
mlb = MultiLabelBinarizer(classes=sorted(CANONICAL_SPICES))
X = np.asarray(mlb.fit_transform(df["spices"]))

print(f"done - {len(df):,} recipes loaded")
print(f"matrix shape: {X.shape}")
print(f"avg spices per recipe: {X.sum(axis=1).mean():.1f}")

loading cookingdataset/RecipeNLG_dataset.csv...
done - 2,231,142 recipes loaded
matrix shape: (2231142, 179)
avg spices per recipe: 1.7


In [13]:
test_data = df.copy()

df.head()

,Unnamed: 0,title,ingredients,directions,link,source,NER,ingredients_parsed,spices
0,0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu...","[brown sugar, milk, vanilla, nuts, butter, bit...",{vanilla}
1,1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom...","[beef, chicken breasts, cream of mushroom soup...",{}
2,2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar...","[frozen corn, cream cheese, butter, garlic pow...","{salt, garlic}"
3,3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroo...","[chicken, chicken gravy, cream of mushroom sou...",{}
4,4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""bu...","[peanut butter, graham cracker crumbs, butter,...",{}


In [14]:
#sample_df.columns

In [15]:
#sample_df.head(15)

In [16]:
print(f"df shape: {df.shape}")
print(f"X shape: {X.shape}")
print(f"spices sample: {list(df['spices'].iloc[0])}")
print(f"non-empty spice rows: {df['spices'].apply(len).gt(0).sum()}")
print(f"avg spices per recipe: {df['spices'].apply(len).mean():.2f}")

df shape: (2231142, 9)
X shape: (2231142, 179)
spices sample: ['vanilla']
non-empty spice rows: 1649782
avg spices per recipe: 1.70


In [17]:
print("first 5 spice sets:")
for i in range(5):
    print(f"  {df['spices'].iloc[i]}")

print()
print("first 5 ingredients_parsed:")
for i in range(5):
    print(f"  {df['ingredients_parsed'].iloc[i][:3]}")

print()
print(f"does df have ingredients_parsed col: {'ingredients_parsed' in df.columns}")
print(f"does df have spices col: {'spices' in df.columns}")
print(f"df columns: {list(df.columns)}")

first 5 spice sets:
  {'vanilla'}
  set()
  {'salt', 'garlic'}
  set()
  set()

first 5 ingredients_parsed:
  ['brown sugar', 'milk', 'vanilla']
  ['beef', 'chicken breasts', 'cream of mushroom soup']
  ['frozen corn', 'cream cheese', 'butter']
  ['chicken', 'chicken gravy', 'cream of mushroom soup']
  ['peanut butter', 'graham cracker crumbs', 'butter']

does df have ingredients_parsed col: True
does df have spices col: True
df columns: ['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source', 'NER', 'ingredients_parsed', 'spices']


In [18]:
from sklearn.cluster       import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics       import silhouette_score

N_CLUSTERS = 100   # bump to 100 for full 2M dataset

# ── cluster ───────────────────────────────────────────────────────────────
# X is (n_recipes, 179) binary matrix from cell 11
# MiniBatchKMeans is the same as KMeans but fast enough for 2M rows

kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS, random_state=42,
    batch_size=4096, n_init=10
)
df["cluster"] = kmeans.fit_predict(X)

# 1. initial clustering
kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS, random_state=42,
    batch_size=4096, n_init=10
)
df["cluster"] = kmeans.fit_predict(X)

# 2. split oversized clusters
SIZE_THRESHOLD = 50_000   # any cluster bigger than this gets split
SUBCLUSTERS    = 5         # how many pieces to break it into
next_id        = N_CLUSTERS  # new sub-cluster IDs start here

for cid in range(N_CLUSTERS):
    mask = df["cluster"] == cid
    if mask.sum() <= SIZE_THRESHOLD:
        continue  # fine as-is, skip

    print(f"splitting cluster {cid} ({mask.sum():,} recipes) into {SUBCLUSTERS}...")
    sub_X      = X[mask.values]
    sub_kmeans = MiniBatchKMeans(
        n_clusters=SUBCLUSTERS, random_state=42,
        batch_size=4096, n_init=5
    )
    sub_labels = sub_kmeans.fit_predict(sub_X)

    # assign new IDs to each sub-cluster
    df.loc[mask, "cluster"] = sub_labels + next_id
    next_id += SUBCLUSTERS

print(f"\ntotal clusters after splitting: {df['cluster'].nunique()}")
print(df["cluster"].value_counts().sort_index().to_string())

print(f"cluster sizes:")
print(df["cluster"].value_counts().sort_index().to_string())

# silhouette score on a 5k sample — tells you how good the clusters are
# >0.2 is reasonable, >0.4 is good
idx = np.random.choice(len(X), min(5000, len(X)), replace=False)
sil = silhouette_score(X[idx], df["cluster"].iloc[idx], metric="jaccard")
print(f"\nsilhouette: {sil:.3f}")

splitting cluster 41 (53,911 recipes) into 5...
splitting cluster 49 (57,286 recipes) into 5...
splitting cluster 63 (582,798 recipes) into 5...
splitting cluster 72 (273,021 recipes) into 5...
splitting cluster 83 (148,669 recipes) into 5...
splitting cluster 84 (111,512 recipes) into 5...

total clusters after splitting: 124
cluster
0       10718
1       16852
2       24343
3        6523
4       13489
5       10300
6        5414
7        4401
8       10749
9        9927
10      12660
11      12127
12      24674
13       6665
14      13454
15      16925
16      10053
17      16477
18       1767
19      14044
20       5987
21      11334
22      11446
23      13830
24      25932
25      13022
26       7294
27      18276
28       9400
29       4081
30      17691
31      43524
32       9838
33      11423
34      17455
35       9635
36       4490
37      18225
38      25957
39      13584
40       4559
42      20656
43      22925
44      10736
45       1151
46       7600
47       9802
48   

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)



silhouette: 0.661


In [19]:
# remove empty clusters from df and remap IDs to be contiguous
df = df[df["cluster"].map(df["cluster"].value_counts()) > 0].reset_index(drop=True)

# remap cluster IDs to 0, 1, 2... with no gaps
old_ids = sorted(df["cluster"].unique())
id_map  = {old: new for new, old in enumerate(old_ids)}
df["cluster"] = df["cluster"].map(id_map)

# also remap cluster_top_spices to match new IDs
if "cluster_top_spices" in globals():
    cluster_top_spices = {id_map[k]: v for k, v in cluster_top_spices.items() if k in id_map}
else:
    cluster_top_spices = {}

print(f"final cluster count: {df['cluster'].nunique()}")
print(f"recipes kept: {len(df):,}")

idx = np.random.choice(len(X), min(5000, len(X)), replace=False)
sil = silhouette_score(X[idx], df["cluster"].iloc[idx], metric="jaccard")
print(f"silhouette after splitting: {sil:.3f}")

final cluster count: 124
recipes kept: 2,231,142


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


silhouette after splitting: 0.669


In [20]:
# what did each cluster find?
# read the top spices per cluster — these describe the flavor group
spice_cols = list(mlb.classes_)
cluster_top_spices = {}

for cid in range(N_CLUSTERS):
    mask    = df["cluster"] == cid
    freq    = X[mask.values].mean(axis=0)
    top_idx = freq.argsort()[::-1][:8]
    top     = [spice_cols[i] for i in top_idx if freq[i] > 0]
    cluster_top_spices[cid] = top
    print(f"cluster {cid:>2}  ({mask.sum():>5} recipes)  {', '.join(top[:6])}")

cluster  0  (10718 recipes)  cinnamon, nutmeg, salt, cloves, garlic, kosher salt
cluster  1  (16852 recipes)  cayenne, rosemary, sage, bay leaf, cardamom, poppy seed
cluster  2  (24343 recipes)  salt, dill, thyme, oregano, cream of tartar, white pepper
cluster  3  ( 6523 recipes)  vanilla, nutmeg, cocoa powder, ginger, mint, cardamom
cluster  4  (13489 recipes)  garlic, cilantro, chili powder, paprika, dill, italian seasoning
cluster  5  (10300 recipes)  salt, vanilla, cream of tartar, kosher salt, cocoa powder, nutmeg
cluster  6  ( 5414 recipes)  oregano, basil, garlic, salt, black pepper, thyme
cluster  7  ( 4401 recipes)  ginger, salt, garlic, cilantro, turmeric, kosher salt
cluster  8  (10749 recipes)  black pepper, salt, mustard, thyme, basil, paprika
cluster  9  ( 9927 recipes)  mustard, salt, parsley, paprika, kosher salt, dill
cluster 10  (12660 recipes)  salt, parsley, garlic, oregano, thyme, bay leaf
cluster 11  (12127 recipes)  garlic, salt, dill, bay leaf, cayenne, italian 

In [47]:
print(df.shape)

(2231142, 10)


In [ ]:



df_sample = df.sample(random_state=42).reset_index(drop=True)

In [ ]:
#Create new dataset with cluster labels

df_sample = df.sample(random_state=42).reset_index(drop=True)

#df.to_csv("SpiceRack/cluster_data.csv", index=False)
print("saved")

saved


In [41]:
df_sample.columns

Index(['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
       'NER', 'ingredients_parsed', 'spices', 'cluster'],
      dtype='object')

In [26]:
N_SVD_DIMS = 100   # bump to 50-100 for full dataset

# compress each recipe from 179 binary dims to 30 dense dims
# after normalize, cosine similarity = dot product (fast)
svd         = TruncatedSVD(n_components=N_SVD_DIMS, random_state=42)
recipe_vecs = svd.fit_transform(X)
recipe_vecs = normalize(recipe_vecs, norm="l2") #this TF-IDF gives each spice a weight based on how rare it is across all recipes. Salt appears in 80% of recipes so it gets a very low weight.

print(f"recipe matrix: {recipe_vecs.shape}")
print(f"variance explained: {svd.explained_variance_ratio_.sum():.1%}")

# save everything the website needs
model = {
    "kmeans":             kmeans,
    "svd":                svd,
    "mlb":                mlb,
    "recipe_matrix":      recipe_vecs,
    "recipe_titles":      df["title"].tolist(),
    "recipe_spices":      [list(s) for s in df["spices"]],
    "cluster_labels":     df["cluster"].tolist(),
    "cluster_top_spices": cluster_top_spices,
    "n_clusters":         N_CLUSTERS,
    "n_recipes":          len(df),
    "silhouette":         round(float(sil), 4),  

}
#joblib.dump(model, "spicerack_model.joblib", compress=3)

#print("model saved.")

print(list(m.keys()))

recipe matrix: (2231142, 100)
variance explained: 100.0%


NameError: name 'm' is not defined

In [ ]:
import joblib
m = joblib.load("models/spicerack_model.joblib")
print("keys:", list(m.keys()))
print("n_recipes:", m["n_recipes"])
print("n_clusters:", m["n_clusters"])
print()

# check a sample recipe
print("recipe_titles sample:", m["recipe_titles"][:3])
print("recipe_spices sample:", m["recipe_spices"][:3])
print("cluster_labels sample:", m["cluster_labels"][:5])
print("recipe_matrix shape:", m["recipe_matrix"].shape)

keys: ['kmeans', 'svd', 'mlb', 'recipe_matrix', 'recipe_titles', 'recipe_spices', 'cluster_labels', 'cluster_top_spices', 'n_clusters', 'n_recipes', 'silhouette']
n_recipes: 2231142
n_clusters: 100

recipe_titles sample: ['No-Bake Nut Cookies', "Jewell Ball'S Chicken", 'Creamy Corn']
recipe_spices sample: [['vanilla'], [], ['salt', 'garlic']]
cluster_labels sample: [118, 108, 97, 108, 108]
recipe_matrix shape: (2231142, 100)


In [ ]:
import numpy as np

cluster_arr = np.array(m["cluster_labels"])
spices_list = m["recipe_spices"]

# check how many recipes have spices in the nearest cluster
test_pantry = {"garlic", "cumin", "paprika", "salt"}
user_bin    = np.asarray(m["mlb"].transform([test_pantry]))
nearest     = int(m["kmeans"].predict(user_bin)[0])

in_cluster  = cluster_arr == nearest
has_spices  = np.array([len(s) > 0 for s in spices_list])

print(f"nearest cluster: {nearest}")
print(f"recipes in cluster: {in_cluster.sum():,}")
print(f"recipes in cluster with spices: {(in_cluster & has_spices).sum():,}")

# try scoring
user_svd = m["svd"].transform(user_bin)[0]
user_svd /= np.linalg.norm(user_svd)
scores   = m["recipe_matrix"] @ user_svd
scores[~in_cluster] = 0
print(f"max score in cluster: {scores.max():.4f}")
print(f"non-zero scores: {(scores > 0).sum():,}")

nearest cluster: 23
recipes in cluster: 13,830
recipes in cluster with spices: 13,830
max score in cluster: 1.0000
non-zero scores: 13,830


---
## - Model functions and Making a Labeled Dataset
Define characteristics of the model created above. Create a labeled dataset for website filtering.



### Initial model creation


In [ ]:
import joblib
import numpy as np

m = joblib.load("spicerack_model.joblib")

# ── step 1: define a test pantry ──────────────────────────────────────────
pantry   = ["garlic", "cumin", "paprika", "salt", "black pepper"]
pantry_set = set(pantry)

# ── step 2: build user vector ─────────────────────────────────────────────
user_bin = np.asarray(m["mlb"].transform([pantry_set]))
user_svd = m["svd"].transform(user_bin)[0]
user_svd = user_svd / np.linalg.norm(user_svd)

print(f"user vector shape: {user_svd.shape}")
print(f"user vector (first 5 values): {user_svd[:5].round(4)}")

# ── step 3: find nearest cluster ──────────────────────────────────────────
nearest = int(m["kmeans"].predict(user_bin)[0])
print(f"\nnearest cluster: {nearest}")
print(f"top spices in that cluster: {m['cluster_top_spices'][nearest]}")

# ── step 4: score every recipe ────────────────────────────────────────────
scores      = m["recipe_matrix"] @ user_svd
cluster_arr = np.array(m["cluster_labels"])
scores[cluster_arr != nearest] = 0

print(f"\nrecipes in cluster: {(cluster_arr == nearest).sum():,}")
print(f"max score: {scores.max():.4f}")
print(f"non-zero scores: {(scores > 0).sum():,}")

# ── step 5: show top 10 ───────────────────────────────────────────────────
top_idx = np.argsort(scores)[::-1][:10]

print(f"\ntop 10 recommendations for pantry {pantry}:")
print(f"{'score':<8} {'title':<45} matched spices")
print("-" * 80)
for i in top_idx:
    if scores[i] <= 0:
        break
    recipe_spices = set(m["recipe_spices"][i])
    matched = sorted(pantry_set & recipe_spices)
    print(f"{scores[i]:.3f}    {m['recipe_titles'][i][:44]:<45} {matched}")

In [41]:
def get_flavor_profiles(pantry, min_coverage=0.3):
    # check how well the users spices cover each flavor profile
    # only returns profiles where they have at least min_coverage fraction of the spices
    results = []
    for profile, spice_set in FLAVOR_PROFILES.items():
        matched = pantry & spice_set
        coverage = len(matched) / len(spice_set)
        if coverage >= min_coverage:
            results.append((profile, round(coverage, 2), sorted(matched)))
    return sorted(results, key=lambda x: -x[1])


def get_regions(profile_names):
    # maps matched flavor profiles to culinary regions
    results = []
    for region, profiles in REGION_PROFILES.items():
        matched = [p for p in profiles if p in profile_names]
        if matched:
            results.append((region, matched))
    return sorted(results, key=lambda x: -len(x[1]))


def recommend(pantry, must_use, top_k=10, min_match=2):
    # jaccard similarity - compares users spice set to each recipes spice set
    # must_use filter removes any recipe that doesnt have all the required spices
    user_vec = np.asarray(mlb.transform([pantry]))

    if must_use:
        spice_cols = list(mlb.classes_)
        must_idx = [spice_cols.index(s) for s in must_use if s in spice_cols]
        if must_idx:
            must_mask = X[:, must_idx].min(axis=1).astype(bool)
        else:
            must_mask = np.ones(len(df), dtype=bool)
    else:
        must_mask = np.ones(len(df), dtype=bool)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sims = 1.0 - pairwise_distances(user_vec, X, metric="jaccard").flatten()

    match_counts = (X & user_vec).sum(axis=1)
    valid = np.where(must_mask & (match_counts >= min_match))[0]

    if len(valid) == 0:
        print("no recipes matched the must-use filter, showing best results without it")
        valid = np.where(match_counts >= min_match)[0]

    ranked = valid[np.lexsort((-match_counts[valid], -sims[valid]))][:top_k]

    out = df.loc[ranked, ["title"]].copy()
    out["similarity"]     = sims[ranked].round(3)
    out["matched_spices"] = df.loc[ranked, "spices"].apply(lambda s: sorted(s & pantry))
    out["num_matched"]    = match_counts[ranked]
    return out.sort_values(["similarity", "num_matched"], ascending=False).reset_index(drop=True)


def tag_region(recipe_spices):
    # assign a single region label to a recipe based on its spice set
    profiles = get_flavor_profiles(recipe_spices, min_coverage=0.2)
    if not profiles:
        return "Other"
    top_profile = profiles[0][0]
    for region, region_profiles in REGION_PROFILES.items():
        if top_profile in region_profiles:
            return region
    return "Other"


def suggest_next_spice(pantry, top_k=5, min_match=2, threshold=0.4):
    # for every spice you dont have, simulate adding it
    # count how many new recipes become available and rank by that
    baseline = recommend(pantry, must_use=set(), top_k=10_000, min_match=min_match)
    baseline_titles = set(baseline["title"])
    baseline_count  = len(baseline[baseline["similarity"] >= threshold])

    missing = [s for s in CANONICAL_SPICES if s not in pantry]
    print(f"baseline: {baseline_count} recipes above {threshold} similarity")
    print(f"testing {len(missing)} candidate spices...")

    results = []
    for spice in missing:
        expanded = pantry | {spice}
        new_recs = recommend(expanded, must_use=set(), top_k=10_000, min_match=min_match)
        new_count  = len(new_recs[new_recs["similarity"] >= threshold])
        new_titles = set(new_recs["title"]) - baseline_titles
        results.append({
            "spice":          spice,
            "newly_unlocked": new_count - baseline_count,
            "examples":       list(new_titles)[:3],
        })

    return pd.DataFrame(results).sort_values("newly_unlocked", ascending=False).head(top_k).reset_index(drop=True)


print("model functions ready")

model functions ready


nearest cluster: 69
cluster top spices: ['marjoram', 'chive', 'mace', 'bay leaf', 'cloves', 'white pepper', 'tarragon', 'sage']
total recipes in cluster: 654
italian seasoning recipes in nearest cluster: 0
italian seasoning recipes across ALL clusters: 13,357


In [38]:
from sklearn.feature_extraction.text import TfidfTransformer

N_SVD_DIMS = 120

# TF-IDF weighting — downweights salt/pepper, upweights rare spices
tfidf   = TfidfTransformer(norm="l2", use_idf=True, smooth_idf=True)
X_tfidf = tfidf.fit_transform(X)

# boost rare spices further by squaring the IDF weights
idf_weights = np.array(tfidf.idf_)
idf_boost   = idf_weights ** 2
idf_boost   = idf_boost / idf_boost.max()
X_boosted   = X_tfidf.multiply(idf_boost)
X_boosted   = normalize(X_boosted, norm="l2")

spice_cols  = list(mlb.classes_)
print(f"salt    weight after boost: {X_boosted[:, spice_cols.index('salt')].mean():.6f}")
print(f"saffron weight after boost: {X_boosted[:, spice_cols.index('saffron')].mean():.6f}")

# SVD compression
svd         = TruncatedSVD(n_components=N_SVD_DIMS, random_state=42)
recipe_vecs = svd.fit_transform(X_boosted)
recipe_vecs = normalize(recipe_vecs, norm="l2")

print(f"recipe matrix: {recipe_vecs.shape}")
print(f"variance explained: {svd.explained_variance_ratio_.sum():.1%}")

# save everything the website needs
model = {
    "kmeans":             kmeans,
    "svd":                svd,
    "mlb":                mlb,
    "tfidf":              tfidf,
    "idf_boost":          idf_boost,
    "recipe_matrix":      recipe_vecs,
    "recipe_titles":      df["title"].tolist(),
    "recipe_spices":      [list(s) for s in df["spices"]],
    "cluster_labels":     df["cluster"].tolist(),
    "cluster_top_spices": cluster_top_spices,
    "n_clusters":         N_CLUSTERS,
    "n_recipes":          len(df),
    "silhouette":         round(float(sil), 4),
}

joblib.dump(model, "spicerack_model2.joblib", compress=3)
print("model saved.")
print("keys:", list(model.keys()))

salt    weight after boost: 0.153267
saffron weight after boost: 0.002192
recipe matrix: (2231142, 120)
variance explained: 100.0%
model saved.
keys: ['kmeans', 'svd', 'mlb', 'tfidf', 'idf_boost', 'recipe_matrix', 'recipe_titles', 'recipe_spices', 'cluster_labels', 'cluster_top_spices', 'n_clusters', 'n_recipes', 'silhouette']


---
### Step - Creating a filtered system
Scripting and classification for basisc filters coded by Austin Pak -
https://github.com/austinpak/Recipes-Restrictions

In [37]:
filter_df = pd.read_csv('SpiceRack-website-main/data/cluster_data.csv')

In [38]:
filter_df.head(2)

,Unnamed: 0,title,ingredients,directions,link,source,NER,ingredients_parsed,spices,cluster
0,0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu...","['brown sugar', 'milk', 'vanilla', 'nuts', 'bu...",{'vanilla'},118
1,1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom...","['beef', 'chicken breasts', 'cream of mushroom...",set(),108


In [34]:
filter_df.columns

Index(['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
       'NER', 'ingredients_parsed', 'spices', 'cluster', 'course_category',
       'is_vegetarian', 'is_vegan', 'allergens_present', 'is_dairy_free',
       'is_gluten_free', 'is_keto', 'is_paleo', 'is_halal', 'is_kosher',
       'is_hindu_friendly'],
      dtype='object')

In [23]:
filter_df.shape

filter_df_copy = filter_df.copy()
filter_df_sample = filter_df_copy.sample(n=1000, random_state=42)

In [ ]:
#filter_df_sample.to_csv('filter_labeled_df_sample.csv', index=False)

---
### New dataset with accurate filter prefrences

The old model was very loose in definitions. New one is more accurate in labeling. 

In [ ]:
import pandas as pd
import re
import ast
import os

# 1. Configuration
INPUT_CSV  = 'SpiceRack/SpiceRack-website-main/data/cluster_data.csv'
REPORT_TXT = 'SpiceRack/SpiceRack-website-main/reports/meat_audit_report.txt'
CHUNK_SIZE = 50000 

# 2. Patterns
meat_keywords = [
    'meat', 'chicken', 'beef', 'pork', 'fish', 'shrimp', 'bacon', 'steak', 
    'tuna', 'salmon', 'lamb', 'turkey', 'sausage', 'ham', 'veal', 'anchovy', 
    'scallop', 'crab', 'lobster', 'oyster', 'clam', 'squid', 'octopus', 
    'duck', 'goose', 'venison', 'gelatin', 'lard', 'hamburger', 'pepperoni', 
    'salami', 'meatball', 'prosciutto', 'mutton', 'chorizo', 'sardine',
    'broth', 'stock', 'bouillon' # Added broth/stock to triggers
]

meat_regex = re.compile(r'\b(' + '|'.join(meat_keywords) + r')\b', re.I)
# The "Gatekeeper": Checks if the broth is specifically vegetable/mushroom/herb
safe_prefix = re.compile(r'\b(vegetable|mushroom|herb|vegan|plant-based)\b', re.I)

def audit_meat(ingredients_str):
    """Returns the specific meat keyword found, or None if safe."""
    try:
        ingredients = ast.literal_eval(str(ingredients_str))
        text = " ".join(ingredients).lower()
    except:
        text = str(ingredients_str).lower()
    
    match = meat_regex.search(text)
    if match:
        trigger = match.group(0).lower()
        
        # --- GATEKEEPER LOGIC ---
        # If the trigger is a broth/stock, check the 15 characters before it for a safe word
        if trigger in ['broth', 'stock', 'bouillon']:
            start = match.start()
            # Look at the snippet of text immediately before the match
            context = text[max(0, start-20):start] 
            if safe_prefix.search(context):
                return None # It's a safe broth, ignore this trigger
        
        return trigger
    return None

# 3. Running the Audit
print(f"🕵️ Analyzing recipes... Generating {REPORT_TXT}")

if not os.path.exists(os.path.dirname(REPORT_TXT)):
    os.makedirs(os.path.dirname(REPORT_TXT))

with open(REPORT_TXT, 'w') as f:
    f.write("MEAT DETECTION AUDIT REPORT (WITH BROTH FILTERING)\n")
    f.write("==================================================\n\n")

    # Verify file exists before reading
    if not os.path.exists(INPUT_CSV):
        print(f"❌ Error: File not found at {INPUT_CSV}")
    else:
        reader = pd.read_csv(INPUT_CSV, chunksize=CHUNK_SIZE)
        total_meat = 0

        for i, chunk in enumerate(reader):
            col = 'NER' if 'NER' in chunk.columns else 'ingredients_parsed'
            
            # Apply updated audit logic
            chunk['detected_meat'] = chunk[col].apply(audit_meat)
            
            # Filter to only meat recipes
            meat_results = chunk[chunk['detected_meat'].notnull()]
            
            for _, row in meat_results.iterrows():
                f.write(f"Title: {row['title']}\n")
                f.write(f"Trigger: {row['detected_meat']}\n")
                f.write(f"Ingredients: {row[col]}\n")
                f.write("-" * 30 + "\n")
            
            total_meat += len(meat_results)
            if (i + 1) % 1 == 0: # Print every chunk for the sample test
                print(f"Processed {(i+1)*len(chunk):,} rows... Found {total_meat:,} meat entries.")

print(f"✨ Audit complete. Total meat recipes found: {total_meat:,}")
print(f"📄 Report saved to {REPORT_TXT}")

🕵️ Analyzing recipes... Generating /Users/daniellarson/Desktop/SpiceRack/SpiceRack-website-main/reports/meat_audit_report.txt
Processed 50,000 rows... Found 15,198 meat entries.
Processed 100,000 rows... Found 30,277 meat entries.
Processed 150,000 rows... Found 45,362 meat entries.
Processed 200,000 rows... Found 60,350 meat entries.
Processed 250,000 rows... Found 75,264 meat entries.
Processed 300,000 rows... Found 90,469 meat entries.
Processed 350,000 rows... Found 105,510 meat entries.
Processed 400,000 rows... Found 120,617 meat entries.
Processed 450,000 rows... Found 135,665 meat entries.
Processed 500,000 rows... Found 150,757 meat entries.
Processed 550,000 rows... Found 165,706 meat entries.
Processed 600,000 rows... Found 180,858 meat entries.
Processed 650,000 rows... Found 195,758 meat entries.
Processed 700,000 rows... Found 210,771 meat entries.
Processed 750,000 rows... Found 225,962 meat entries.
Processed 800,000 rows... Found 241,015 meat entries.
Processed 850,000

In [34]:
import pandas as pd
import re
import ast
import os

# CONFIGURATION
INPUT_CSV = 'SpiceRack-website-main/data/cluster_data.csv' # Your 2.4GB file
SUMMARY_CSV = 'SpiceRack-website-main/reports/meat_audit_report.csv'
CHUNK_SIZE = 100000 

# REGEX PATTERNS
meat_keywords = [
    'meat', 'chicken', 'beef', 'pork', 'fish', 'shrimp', 'bacon', 'steak', 
    'tuna', 'salmon', 'lamb', 'turkey', 'sausage', 'ham', 'veal', 'anchovy', 
    'scallop', 'crab', 'lobster', 'oyster', 'clam', 'squid', 'octopus', 
    'duck', 'goose', 'venison', 'gelatin', 'lard', 'hamburger', 'pepperoni', 
    'salami', 'meatball', 'prosciutto', 'mutton', 'chorizo', 'sardine',
    'broth', 'stock', 'bouillon'
]
meat_regex = re.compile(r'\b(' + '|'.join(meat_keywords) + r')\b', re.I)
safe_prefix = re.compile(r'\b(vegetable|mushroom|herb|vegan|plant-based)\b', re.I)

def audit_meat(ingredients_str):
    try:
        # Parsing NER or ingredients list
        text = " ".join(ast.literal_eval(str(ingredients_str))).lower()
    except:
        text = str(ingredients_str).lower()
    
    match = meat_regex.search(text)
    if match:
        trigger = match.group(0).lower()
        # Gatekeeper Logic for Broths
        if trigger in ['broth', 'stock', 'bouillon']:
            context = text[max(0, match.start()-20):match.start()] 
            if safe_prefix.search(context):
                return None 
        return trigger
    return None

# EXECUTION
if os.path.exists(SUMMARY_CSV): os.remove(SUMMARY_CSV)

print(f"🕵️ Scanning 2.2M recipes...")
reader = pd.read_csv(INPUT_CSV, chunksize=CHUNK_SIZE)
total_meat = 0

for i, chunk in enumerate(reader):
    col = 'NER' if 'NER' in chunk.columns else 'ingredients_parsed'
    chunk['detected_meat'] = chunk[col].apply(audit_meat)
    
    # Save only the 'Meat' rows to a smaller audit file
    meat_only = chunk[chunk['detected_meat'].notnull()]
    meat_only.to_csv(SUMMARY_CSV, mode='a', index=False, header=(i==0))
    
    total_meat += len(meat_only)
    print(f"Chunk {i+1}: Found {total_meat:,} meat entries so far...")

print(f"✨ Audit Summary saved to {SUMMARY_CSV}")

🕵️ Scanning 2.2M recipes...
Chunk 1: Found 30,277 meat entries so far...
Chunk 2: Found 60,350 meat entries so far...
Chunk 3: Found 90,469 meat entries so far...
Chunk 4: Found 120,617 meat entries so far...
Chunk 5: Found 150,757 meat entries so far...
Chunk 6: Found 180,858 meat entries so far...
Chunk 7: Found 210,771 meat entries so far...
Chunk 8: Found 241,015 meat entries so far...
Chunk 9: Found 271,212 meat entries so far...
Chunk 10: Found 306,024 meat entries so far...
Chunk 11: Found 343,202 meat entries so far...
Chunk 12: Found 380,857 meat entries so far...
Chunk 13: Found 418,367 meat entries so far...
Chunk 14: Found 449,607 meat entries so far...
Chunk 15: Found 481,469 meat entries so far...
Chunk 16: Found 507,905 meat entries so far...
Chunk 17: Found 544,596 meat entries so far...
Chunk 18: Found 581,220 meat entries so far...
Chunk 19: Found 617,874 meat entries so far...
Chunk 20: Found 654,653 meat entries so far...
Chunk 21: Found 691,202 meat entries so far.

In [35]:
# Load the much smaller summary file
audit_df = pd.read_csv('SpiceRack-website-main/reports/meat_audit_report.csv')

# 1. Check for "The Graham Cracker Problem"
# Search for recipes triggered by 'ham' that are actually crackers
ham_check = audit_df[(audit_df['detected_meat'] == 'ham') & 
                     (audit_df['title'].str.contains('Graham', case=False))]

print(f"Found {len(ham_check)} Graham Cracker errors:")
display(ham_check[['title', 'detected_meat', 'NER']].head(10))

# 2. Check for "The Vegetable Broth Problem"
# Search for recipes where the 'Gatekeeper' might have failed
broth_check = audit_df[(audit_df['detected_meat'].isin(['broth', 'stock'])) & 
                       (audit_df['title'].str.contains('Vegetable', case=False))]

print(f"\nFound {len(broth_check)} Vegetable Broth potential errors:")
display(broth_check[['title', 'detected_meat', 'NER']].head(10))

# 3. Top Trigger Words
print("\nMost frequent meat triggers:")
print(audit_df['detected_meat'].value_counts().head(20))

Found 2 Graham Cracker errors:


,title,detected_meat,NER
156304,Ham Graham Loaf,ham,"[""ham"", ""fresh lean pork"", ""egg"", ""crackers"", ..."
604364,Black Bean Soup Graham Kerr's Smart Cooking Re...,ham,"[""Black turtle beans"", ""Ham hocks"", ""Sesame oi..."



Found 135 Vegetable Broth potential errors:


,title,detected_meat,NER
2211,Fat Free Vegetable Soup,broth,"[""water"", ""packets Washington brown seasoning ..."
13060,Wanda'S Vegetable Soup,broth,"[""broth"", ""onion"", ""celery"", ""green pepper"", ""..."
15563,Vegetable Soup(Ukrainian),stock,"[""onion"", ""butter"", ""potato"", ""vegetables"", ""g..."
21091,Vegetable Soup,broth,"[""olive oil"", ""onion"", ""garlic"", ""green beans""..."
23022,Vegetable Curry,stock,"[""cauliflower"", ""tomatoes"", ""potatoes"", ""shell..."
35090,Vegetable Soup,stock,"[""soup stock"", ""oil"", ""onion"", ""celery"", ""carr..."
80611,Vegetable Soup,stock,"[""stock"", ""tomatoes"", ""celery"", ""carrots"", ""gr..."
94605,Chicken Vegetable Noodle Soup,broth,"[""broth"", ""onions"", ""carrots"", ""stalks celery""..."
111797,The Best Vegetable Soup,broth,"[""onion"", ""celery"", ""butter"", ""carrots"", ""pota..."
112203,Vegetable Pasta Soup,broth,"[""clear broth"", ""basil"", ""garlic powder"", ""tom..."



Most frequent meat triggers:
detected_meat
chicken      237543
beef         127840
bacon         63431
pork          51902
shrimp        32757
sausage       29153
ham           25151
turkey        24392
hamburger     18902
gelatin       17729
salmon        17615
tuna          11731
fish          11354
meat          10331
lamb           8750
steak          8304
crab           6261
pepperoni      4606
anchovy        4142
veal           4035
Name: count, dtype: int64


In [37]:
import pandas as pd
import re
import ast

# 1. Patterns (The 'Gold Standard' logic we refined)
meat_pattern = re.compile(r'\b(meat|chicken|beef|pork|fish|shrimp|bacon|steak|tuna|salmon|lamb|turkey|sausage|ham|veal|anchovy|scallop|crab|lobster|oyster|clam|squid|octopus|duck|goose|venison|gelatin|lard|hamburger|pepperoni|salami|meatball|prosciutto)\b', re.I)
stock_keywords = re.compile(r'\b(broth|stock|bouillon)\b', re.I)
safe_prefix = re.compile(r'\b(vegetable|mushroom|herb|vegan|plant-based)\b', re.I)

def fast_parser(row):
    title = str(row['title']).lower()
    try:
        ingredients = ast.literal_eval(str(row['NER']))
        text = "  ".join(ingredients).lower()
    except:
        text = str(row['NER']).lower()

    # Meat Check
    has_meat = bool(meat_pattern.search(text))
    
    # Stock Check (with Title-Awareness)
    has_animal_stock = False
    stock_match = stock_keywords.search(text)
    if stock_match:
        is_explicitly_safe = bool(safe_prefix.search(text[max(0, stock_match.start()-20):stock_match.start()]))
        is_title_safe = bool(safe_prefix.search(title))
        if not (is_explicitly_safe or is_title_safe):
            has_animal_stock = True

    is_vegetarian = not (has_meat or has_animal_stock)
    
    # Vegan Check
    has_dairy = bool(re.search(r'\b(milk|cheese|butter|yogurt|cream|whey|ghee|parmesan)\b', text))
    has_eggs = bool(re.search(r'\b(egg|eggs|mayo)\b', text))
    is_vegan = is_vegetarian and not (has_dairy or has_eggs or "honey" in text)

    return pd.Series({'is_vegetarian': is_vegetarian, 'is_vegan': is_vegan})

# 2. Sample & Test (No Output File Created)
# We load just a random sample to verify the logic
print("🔄 Loading sample from 2.2M dataset...")
df_sample = pd.read_csv('SpiceRack-website-main/data/cluster_data.csv', nrows=10000).sample(2000)

print("🧪 Running validation logic...")
results = df_sample.apply(fast_parser, axis=1)
df_val = pd.concat([df_sample, results], axis=1)

# 3. Direct Notebook Display (The 'Invisible' Report)
print("\n✅ VALIDATION COMPLETE")
print("-" * 30)

# Show me the 'Meat' recipes it found (to ensure they are actually meat)
print("TOP MEAT SAMPLES FOUND:")
display(df_val[df_val['is_vegetarian'] == False][['title', 'is_vegetarian']].head(10))

# Show me 'Vegetable' titles it marked as Vegetarian (to ensure Gatekeeper worked)
print("\nGATEKEEPER CHECK (Vegetable titles):")
display(df_val[df_val['title'].str.contains('Vegetable', case=False)][['title', 'is_vegetarian']].head(10))

🔄 Loading sample from 2.2M dataset...
🧪 Running validation logic...

✅ VALIDATION COMPLETE
------------------------------
TOP MEAT SAMPLES FOUND:


,title,is_vegetarian
5297,Sweet And Sour Chicken Stir-Fry,False
377,Double Baked Potatoes,False
49,Vegetable-Burger Soup,False
551,Easy Hot Dish,False
8844,Hamburger Corn Casserole,False
203,Nana'S Cornbread(For 9-Inch Iron Skillet Or 8 ...,False
9093,Low Country Boil(South Carolina),False
3581,Surprise Sandwich Loaf,False
670,Cranberry Souffle,False
7543,Chicken Parmesan,False



GATEKEEPER CHECK (Vegetable titles):


,title,is_vegetarian
49,Vegetable-Burger Soup,False
8856,Stir-Fried Beef With Vegetables,False
3982,Chicken Soup With Chinese Vegetables,False
471,Vegetable Pizza,True
7809,Swiss Vegetable Medley,True
3594,Vegetable Dip,True
8404,Hamburger Vegetable Soup,True
4767,Aunty Ann'S Vegetable Chowder,True
484,No Fat Rice And Vegetable Dish,True
3230,Vegetable Rice Salad,True


In [35]:
filter_df.columns

Index(['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
       'NER', 'ingredients_parsed', 'spices', 'cluster', 'course_category',
       'is_vegetarian', 'is_vegan', 'allergens_present', 'is_dairy_free',
       'is_gluten_free', 'is_keto', 'is_paleo', 'is_halal', 'is_kosher',
       'is_hindu_friendly'],
      dtype='object')

In [39]:
import pandas as pd
import re
import ast
import os

# ==========================================
# 1. CONFIGURATION
# ==========================================
INPUT_FILE = 'SpiceRack-website-main/data/cluster_data.csv' 
OUTPUT_FILE = 'SpiceRack-website-main/data/full_recipes_with_restrictions.csv'
CHUNK_SIZE = 50000 

# Regex patterns for high-accuracy detection
meat_pat = re.compile(r'\b(meat|chicken|beef|pork|fish|shrimp|bacon|steak|tuna|salmon|lamb|turkey|sausage|ham|veal|anchovy|scallop|crab|lobster|oyster|clam|squid|octopus|duck|goose|venison|gelatin|lard|hamburger|pepperoni|salami|meatball|prosciutto|chorizo|sardine)\b', re.I)
stock_pat = re.compile(r'\b(broth|stock|bouillon)\b', re.I)
safe_pat = re.compile(r'\b(vegetable|mushroom|herb|vegan|plant-based)\b', re.I)
dairy_pat = re.compile(r'\b(milk|cheese|butter|yogurt|cream|whey|ghee|parmesan|mozzarella|cheddar|ricotta|feta|paneer)\b', re.I)
egg_pat = re.compile(r'\b(egg|eggs|mayo|mayonnaise)\b', re.I)
sugar_pat = re.compile(r'\b(sugar|honey|syrup|corn\s+starch|flour|potato|rice|pasta|bread|beans|legumes)\b', re.I)
paleo_bad_pat = re.compile(r'\b(dairy|milk|cheese|sugar|flour|bread|pasta|rice|corn|legumes|soy|peanuts)\b', re.I)

def master_recipe_parser(row):
    title = str(row['title']).lower()
    col = 'NER' if 'NER' in row.index else 'ingredients_parsed'
    try:
        ingredients_list = ast.literal_eval(str(row[col]))
        text = "  ".join(ingredients_list).lower()
    except:
        text = str(row[col]).lower()

    # 1. Vegetarian & Vegan (with Title-Awareness)
    has_meat = bool(meat_pat.search(text))
    has_animal_stock = False
    stock_match = stock_pat.search(text)
    if stock_match:
        if not (safe_pat.search(text[max(0, stock_match.start()-20):stock_match.start()]) or safe_pat.search(title)):
            has_animal_stock = True
    
    is_vegetarian = not (has_meat or has_animal_stock)
    has_dairy = bool(dairy_pat.search(text))
    has_eggs = bool(egg_pat.search(text))
    is_vegan = is_vegetarian and not (has_dairy or has_eggs or "honey" in text)

    # 2. Allergens & Lifestyle
    allergens = []
    if has_dairy: allergens.append("Dairy")
    if has_eggs: allergens.append("Eggs")
    if "peanut" in text: allergens.append("Peanuts")
    if "shrimp" in text or "crab" in text or "lobster" in text: allergens.append("Shellfish")
    
    is_keto = not bool(sugar_pat.search(text))
    is_paleo = not bool(paleo_bad_pat.search(text))
    is_gluten_free = not bool(re.search(r'\b(wheat|flour|barley|rye|couscous|semolina)\b', text))

    # 3. Religious
    has_pork = bool(re.search(r'\b(pork|ham|bacon|lard|prosciutto|chorizo)\b', text))
    has_shellfish = "Shellfish" in allergens
    has_alcohol = bool(re.search(r'\b(wine|beer|vodka|rum|alcohol)\b', text))

    return pd.Series({
        'is_vegetarian': is_vegetarian,
        'is_vegan': is_vegan,
        'allergens_present': ", ".join(allergens) if allergens else "None",
        'is_dairy_free': not has_dairy,
        'is_gluten_free': is_gluten_free,
        'is_keto': is_keto,
        'is_paleo': is_paleo,
        'is_halal': not (has_pork or has_alcohol),
        'is_kosher': not (has_pork or has_shellfish),
        'is_hindu_friendly': not (bool(re.search(r'\b(beef|steak|hamburger|gelatin)\b', text)))
    })

# ==========================================
# 2. PROCESSING LOOP (Enforces Column Order)
# ==========================================
if os.path.exists(OUTPUT_FILE): os.remove(OUTPUT_FILE)

reader = pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)

for i, chunk in enumerate(reader):
    # Apply logic
    new_cols = chunk.apply(master_recipe_parser, axis=1)
    processed = pd.concat([chunk, new_cols], axis=1)
    
    # Force EXACT column order requested
    final_cols = [
        'Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
        'NER', 'ingredients_parsed', 'spices', 'cluster', 'course_category',
        'is_vegetarian', 'is_vegan', 'allergens_present', 'is_dairy_free',
        'is_gluten_free', 'is_keto', 'is_paleo', 'is_halal', 'is_kosher',
        'is_hindu_friendly'
    ]
    
    # If some columns like 'Unnamed: 0' are missing in input, create them
    for col in final_cols:
        if col not in processed.columns:
            processed[col] = "None" if "is_" not in col else False

    # Save to CSV
    processed[final_cols].to_csv(OUTPUT_FILE, mode='a', index=False, header=(i==0))
    print(f"✅ Chunk {i+1} complete.")

✅ Chunk 1 complete.
✅ Chunk 2 complete.
✅ Chunk 3 complete.
✅ Chunk 4 complete.
✅ Chunk 5 complete.
✅ Chunk 6 complete.
✅ Chunk 7 complete.
✅ Chunk 8 complete.
✅ Chunk 9 complete.
✅ Chunk 10 complete.
✅ Chunk 11 complete.
✅ Chunk 12 complete.
✅ Chunk 13 complete.
✅ Chunk 14 complete.
✅ Chunk 15 complete.
✅ Chunk 16 complete.
✅ Chunk 17 complete.
✅ Chunk 18 complete.
✅ Chunk 19 complete.
✅ Chunk 20 complete.
✅ Chunk 21 complete.
✅ Chunk 22 complete.
✅ Chunk 23 complete.
✅ Chunk 24 complete.
✅ Chunk 25 complete.
✅ Chunk 26 complete.
✅ Chunk 27 complete.
✅ Chunk 28 complete.
✅ Chunk 29 complete.
✅ Chunk 30 complete.
✅ Chunk 31 complete.
✅ Chunk 32 complete.
✅ Chunk 33 complete.
✅ Chunk 34 complete.
✅ Chunk 35 complete.
✅ Chunk 36 complete.
✅ Chunk 37 complete.
✅ Chunk 38 complete.
✅ Chunk 39 complete.
✅ Chunk 40 complete.
✅ Chunk 41 complete.
✅ Chunk 42 complete.
✅ Chunk 43 complete.
✅ Chunk 44 complete.
✅ Chunk 45 complete.


In [40]:
import pandas as pd
import re
import ast
import os

# ==========================================
# 1. CONFIGURATION
# ==========================================
INPUT_FILE = 'SpiceRack-website-main/data/cluster_data.csv' 
OUTPUT_FILE = 'SpiceRack-website-main/data/full_recipes_with_restrictions.csv'
CHUNK_SIZE = 50000 

# Regex patterns for high-accuracy detection
meat_pat = re.compile(r'\b(meat|chicken|beef|pork|fish|shrimp|bacon|steak|tuna|salmon|lamb|turkey|sausage|ham|veal|anchovy|scallop|crab|lobster|oyster|clam|squid|octopus|duck|goose|venison|gelatin|lard|hamburger|pepperoni|salami|meatball|prosciutto|chorizo|sardine)\b', re.I)
stock_pat = re.compile(r'\b(broth|stock|bouillon)\b', re.I)
safe_pat = re.compile(r'\b(vegetable|mushroom|herb|vegan|plant-based)\b', re.I)
dairy_pat = re.compile(r'\b(milk|cheese|butter|yogurt|cream|whey|ghee|parmesan|mozzarella|cheddar|ricotta|feta|paneer)\b', re.I)
egg_pat = re.compile(r'\b(egg|eggs|mayo|mayonnaise)\b', re.I)
sugar_pat = re.compile(r'\b(sugar|honey|syrup|corn\s+starch|flour|potato|rice|pasta|bread|beans|legumes)\b', re.I)
paleo_bad_pat = re.compile(r'\b(dairy|milk|cheese|sugar|flour|bread|pasta|rice|corn|legumes|soy|peanuts)\b', re.I)

def master_recipe_parser(row):
    # DUAL-SCOPE: We pull both Title and Ingredients
    title_text = str(row['title']).lower()
    
    col = 'NER' if 'NER' in row.index else 'ingredients_parsed'
    try:
        ingredients_list = ast.literal_eval(str(row[col]))
        ing_text = "  ".join(ingredients_list).lower()
    except:
        ing_text = str(row[col]).lower()

    # 1. VEGETARIAN LOGIC (Dual-Scope Search)
    # Checks ingredients AND title for meat keywords (fixes Pork Au Poivre)
    has_meat = bool(meat_pat.search(ing_text)) or bool(meat_pat.search(title_text))
    
    # Title-Aware Broth logic
    has_animal_stock = False
    stock_match = stock_pat.search(ing_text)
    if stock_match:
        # If broth is found, ignore if 'vegetable' is in context or in the title
        if not (safe_pat.search(ing_text[max(0, stock_match.start()-20):stock_match.start()]) or safe_pat.search(title_text)):
            has_animal_stock = True
    
    is_vegetarian = not (has_meat or has_animal_stock)
    
    has_dairy = bool(dairy_pat.search(ing_text))
    has_eggs = bool(egg_pat.search(ing_text))
    is_vegan = is_vegetarian and not (has_dairy or has_eggs or "honey" in ing_text)

    # 2. ALLERGENS & LIFESTYLE
    allergens = []
    if has_dairy: allergens.append("Dairy")
    if has_eggs: allergens.append("Eggs")
    if "peanut" in ing_text: allergens.append("Peanuts")
    if "shrimp" in ing_text or "crab" in ing_text or "lobster" in ing_text: allergens.append("Shellfish")
    
    is_keto = not bool(sugar_pat.search(ing_text))
    is_paleo = not bool(paleo_bad_pat.search(ing_text))
    is_gluten_free = not bool(re.search(r'\b(wheat|flour|barley|rye|couscous|semolina)\b', ing_text))

    # 3. RELIGIOUS (Dual-Scope)
    has_pork = bool(re.search(r'\b(pork|ham|bacon|lard|prosciutto|chorizo)\b', ing_text)) or \
               bool(re.search(r'\b(pork|ham|bacon|lard|prosciutto|chorizo)\b', title_text))
               
    has_shellfish = "Shellfish" in allergens or bool(re.search(r'\b(shrimp|crab|lobster|oyster|clam|scallop|mussel)\b', title_text))
    has_alcohol = bool(re.search(r'\b(wine|beer|vodka|rum|alcohol)\b', ing_text)) or \
                  bool(re.search(r'\b(wine|beer|vodka|rum|alcohol)\b', title_text))

    return pd.Series({
        'is_vegetarian': is_vegetarian,
        'is_vegan': is_vegan,
        'allergens_present': ", ".join(allergens) if allergens else "None",
        'is_dairy_free': not has_dairy,
        'is_gluten_free': is_gluten_free,
        'is_keto': is_keto,
        'is_paleo': is_paleo,
        'is_halal': not (has_pork or has_alcohol),
        'is_kosher': not (has_pork or has_shellfish),
        'is_hindu_friendly': not (bool(re.search(r'\b(beef|steak|hamburger|gelatin)\b', ing_text)) or \
                                 bool(re.search(r'\b(beef|steak|hamburger|gelatin)\b', title_text)))
    })

# ==========================================
# 2. PROCESSING LOOP (Ensures Spices & Order)
# ==========================================
if os.path.exists(OUTPUT_FILE): os.remove(OUTPUT_FILE)

reader = pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)

for i, chunk in enumerate(reader):
    # Apply logic to the WHOLE ROW (axis=1) for dual-scope searching
    new_cols = chunk.apply(master_recipe_parser, axis=1)
    
    # CRITICAL: Merge instead of concat to ensure 'spices' stays aligned
    processed = chunk.merge(new_cols, left_index=True, right_index=True)
    
    # Force EXACT 21-column order requested
    final_cols = [
        'Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
        'NER', 'ingredients_parsed', 'spices', 'cluster', 'course_category',
        'is_vegetarian', 'is_vegan', 'allergens_present', 'is_dairy_free',
        'is_gluten_free', 'is_keto', 'is_paleo', 'is_halal', 'is_kosher',
        'is_hindu_friendly'
    ]
    
    # Fill missing columns (only if they weren't in original file)
    for col in final_cols:
        if col not in processed.columns:
            processed[col] = "" if "is_" not in col else False

    # Save to CSV
    processed[final_cols].to_csv(OUTPUT_FILE, mode='a', index=False, header=(i==0))
    print(f"✅ Chunk {i+1} complete. Processed {(i+1)*len(chunk):,} recipes so far.")
    filter_test = pd.read_csv(OUTPUT_FILE)  

✅ Chunk 1 complete. Processed 50,000 recipes so far.
✅ Chunk 2 complete. Processed 100,000 recipes so far.
✅ Chunk 3 complete. Processed 150,000 recipes so far.
✅ Chunk 4 complete. Processed 200,000 recipes so far.
✅ Chunk 5 complete. Processed 250,000 recipes so far.
✅ Chunk 6 complete. Processed 300,000 recipes so far.
✅ Chunk 7 complete. Processed 350,000 recipes so far.
✅ Chunk 8 complete. Processed 400,000 recipes so far.
✅ Chunk 9 complete. Processed 450,000 recipes so far.
✅ Chunk 10 complete. Processed 500,000 recipes so far.
✅ Chunk 11 complete. Processed 550,000 recipes so far.
✅ Chunk 12 complete. Processed 600,000 recipes so far.
✅ Chunk 13 complete. Processed 650,000 recipes so far.
✅ Chunk 14 complete. Processed 700,000 recipes so far.
✅ Chunk 15 complete. Processed 750,000 recipes so far.
✅ Chunk 16 complete. Processed 800,000 recipes so far.
✅ Chunk 17 complete. Processed 850,000 recipes so far.
✅ Chunk 18 complete. Processed 900,000 recipes so far.
✅ Chunk 19 complete.

In [43]:
filter_test = pd.read_csv(OUTPUT_FILE) 

In [44]:
filter_test.columns. tolist()

['Unnamed: 0',
 'title',
 'ingredients',
 'directions',
 'link',
 'source',
 'NER',
 'ingredients_parsed',
 'spices',
 'cluster',
 'course_category',
 'is_vegetarian',
 'is_vegan',
 'allergens_present',
 'is_dairy_free',
 'is_gluten_free',
 'is_keto',
 'is_paleo',
 'is_halal',
 'is_kosher',
 'is_hindu_friendly']

In [45]:
filter_test.head(2) 

,Unnamed: 0,title,ingredients,directions,link,source,NER,ingredients_parsed,spices,cluster,...,is_vegetarian,is_vegan,allergens_present,is_dairy_free,is_gluten_free,is_keto,is_paleo,is_halal,is_kosher,is_hindu_friendly
0,0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu...","['brown sugar', 'milk', 'vanilla', 'nuts', 'bu...",{'vanilla'},118,...,True,False,Dairy,False,True,False,False,True,True,True
1,1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom...","['beef', 'chicken breasts', 'cream of mushroom...",set(),108,...,False,False,Dairy,False,True,True,True,True,True,False


In [42]:
filter_test.shape

(2231142, 21)

In [ ]:
#filter_test.head(1000).to_csv('SpiceRack-website-main/data/full_recipes_with_restrictions_sample.csv', index=False)

---
## - Test Results of Modal 
Backend and Notebook only results that shows the modal trained works.

In [ ]:
#old code, old cell. 

# If you are reading this please hire me :) 
# www.linkedin.com/in/daniellarson1

# validate the inputs from step 1
"""pantry   = validate_spices(MY_PANTRY, label="pantry")
must_use = validate_spices(MUST_USE,  label="must-use")

extras = must_use - pantry
if extras:
    print(f"added to pantry automatically: {extras}")
    pantry |= extras

print(f"\npantry:   {sorted(pantry)}")
print(f"must-use: {sorted(must_use)}")


# --- flavor profiles ---
print("\n" + "-" * 55)
print("flavor profiles")
print("-" * 55)

profiles = get_flavor_profiles(pantry)
if not profiles:
    print("no strong profiles found - try adding more spices")
else:
    for name, score, matched in profiles:
        bar = "#" * int(score * 10) + "." * (10 - int(score * 10))
        print(f"  [{bar}] {int(score*100)}%  {name}")
        print(f"           spices: {', '.join(matched)}")


# --- culinary regions ---
print("\n" + "-" * 55)
print("cuisines you can cook")
print("-" * 55)

regions = get_regions([p[0] for p in profiles])
if not regions:
    print("no strong regional match found")
else:
    for region, matched_profiles in regions:
        print(f"  {region}")
        print(f"    via: {', '.join(matched_profiles)}")


# --- top recipe recommendations ---
print("\n" + "-" * 55)
if must_use:
    print(f"top recipes (must contain: {sorted(must_use)})")
else:
    print("top recipes")
print("-" * 55)

recs = recommend(pantry, must_use, top_k=10, min_match=2)
print(recs[["title", "similarity", "matched_spices", "num_matched"]].to_string(index=False))


# --- recipes grouped by region ---
print("\n" + "-" * 55)
print("recipes by region")
print("-" * 55)

pool = recommend(pantry, must_use, top_k=200, min_match=2)
pool["region"] = pool["title"].apply(
    lambda t: tag_region(
        df[df["title"] == t]["spices"].iloc[0]
        if (df["title"] == t).any() else set()
    )
)

for region in pool["region"].unique():
    if region == "Other":
        continue
    subset = pool[pool["region"] == region].head(3)
    print(f"\n  {region}")
    for _, row in subset.iterrows():
        print(f"    - {row['title']}  (similarity: {row['similarity']})")


# --- what spice should you buy next ---
print("\n" + "-" * 55)
print("spices to buy next")
print("-" * 55)

next_spice = suggest_next_spice(pantry, top_k=5, min_match=2)
for _, row in next_spice.iterrows():
    print(f"  {row['spice']:<25} {row['newly_unlocked']:+} new recipes")
    if row["examples"]:
        print(f"  {'':25} e.g. {row['examples'][0]}") """ 


pantry:   ['black pepper', 'chili powder', 'cinnamon', 'cumin', 'garlic', 'ginger', 'oregano', 'paprika', 'salt']
must-use: ['cumin', 'garlic']

-------------------------------------------------------
flavor profiles
-------------------------------------------------------
  [###.......] 31%  American BBQ & Southern
           spices: black pepper, chili powder, cumin, garlic, paprika

-------------------------------------------------------
cuisines you can cook
-------------------------------------------------------
  American BBQ / Southern
    via: American BBQ & Southern

-------------------------------------------------------
top recipes (must contain: ['cumin', 'garlic'])
-------------------------------------------------------
                                                    title  similarity                                                      matched_spices  num_matched
                                          Chicken Karrahi       0.500                         [black peppe

### 



In [27]:
import joblib
import numpy as np

m = joblib.load("models/spicerack_model.joblib")

# ── step 1: define a test pantry ──────────────────────────────────────────
pantry   = ["garlic", "cumin", "paprika", "salt", "black pepper"]
pantry_set = set(pantry)

# ── step 2: build user vector ─────────────────────────────────────────────
user_bin = np.asarray(m["mlb"].transform([pantry_set]))
user_svd = m["svd"].transform(user_bin)[0]
user_svd = user_svd / np.linalg.norm(user_svd)

print(f"user vector shape: {user_svd.shape}")
print(f"user vector (first 5 values): {user_svd[:5].round(4)}")

# ── step 3: find nearest cluster ──────────────────────────────────────────
nearest = int(m["kmeans"].predict(user_bin)[0])
print(f"\nnearest cluster: {nearest}")
print(f"top spices in that cluster: {m['cluster_top_spices'][nearest]}")

# ── step 4: score every recipe ────────────────────────────────────────────
scores      = m["recipe_matrix"] @ user_svd
cluster_arr = np.array(m["cluster_labels"])
scores[cluster_arr != nearest] = 0

print(f"\nrecipes in cluster: {(cluster_arr == nearest).sum():,}")
print(f"max score: {scores.max():.4f}")
print(f"non-zero scores: {(scores > 0).sum():,}")

# ── step 5: show top 10 ───────────────────────────────────────────────────
top_idx = np.argsort(scores)[::-1][:10]

print(f"\ntop 10 recommendations for pantry {pantry}:")
print(f"{'score':<8} {'title':<45} matched spices")
print("-" * 80)
for i in top_idx:
    if scores[i] <= 0:
        break
    recipe_spices = set(m["recipe_spices"][i])
    matched = sorted(pantry_set & recipe_spices)
    print(f"{scores[i]:.3f}    {m['recipe_titles'][i][:44]:<45} {matched}")

user vector shape: (100,)
user vector (first 5 values): [ 0.6639  0.3138  0.0824 -0.0093  0.1666]

nearest cluster: 23
top spices in that cluster: ['paprika', 'garlic', 'salt', 'black pepper', 'cayenne', 'onion powder', 'oregano', 'thyme']

recipes in cluster: 13,830
max score: 1.0000
non-zero scores: 13,830

top 10 recommendations for pantry ['garlic', 'cumin', 'paprika', 'salt', 'black pepper']:
score    title                                         matched spices
--------------------------------------------------------------------------------
1.000    Bake-Fried Potatoes                           ['black pepper', 'cumin', 'garlic', 'paprika', 'salt']
1.000    Chickaritos                                   ['black pepper', 'cumin', 'garlic', 'paprika', 'salt']
1.000    Lighter Version of Yotam Ottolenghi & Sami T  ['black pepper', 'cumin', 'garlic', 'paprika', 'salt']
1.000    Classic Hummus with Spiced 'n Baked Pita Chi  ['black pepper', 'cumin', 'garlic', 'paprika', 'salt']
1.000   

---
## Debugging 
For testing and errors related to the project and model making. Not relevant to the core project structure.

In [9]:
# Recipe lookup for its properties 

recipe_lookup_df = pd.read_csv('SpiceRack-website-main/data/full_recipes_with_restrictions.csv')

# Rename 'Old_Name' to 'New_Name'
recipe_lookup_df.rename(columns={'title': 'Recipe Name'}, inplace=True)

recipe_lookup_df.columns

Index(['Unnamed: 0', 'Recipe Name', 'ingredients', 'directions', 'link',
       'source', 'NER', 'ingredients_parsed', 'spices', 'cluster',
       'course_category', 'is_vegetarian', 'is_vegan', 'allergens_present',
       'is_dairy_free', 'is_gluten_free', 'is_keto', 'is_paleo', 'is_halal',
       'is_kosher', 'is_hindu_friendly'],
      dtype='object')

In [3]:
test = pd.read_csv('SpiceRack-website-main/data/full_recipes_with_restrictions_sample.csv')

test.columns

Index(['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
       'NER', 'ingredients_parsed', 'spices', 'cluster', 'course_category',
       'is_vegetarian', 'is_vegan', 'allergens_present', 'is_dairy_free',
       'is_gluten_free', 'is_keto', 'is_paleo', 'is_halal', 'is_kosher',
       'is_hindu_friendly'],
      dtype='object')

In [6]:
#Input the recipe name you want to look up
recipe_name = "Low-Carb Glazed Ribs"

In [ ]:

# Recipe lookup for its properties 

cols = ['Recipe Name', 'is_vegetarian', 'is_vegan', 'allergens_present', 'is_dairy_free',
       'is_gluten_free', 'is_keto', 'is_paleo', 'is_halal', 'is_kosher',
       'is_hindu_friendly']

results = recipe_lookup_df.loc[recipe_lookup_df["Recipe Name"] == recipe_name]
cols = [c for c in cols if c in results.columns]
results = recipe_lookup_df.loc[recipe_lookup_df["Recipe Name"] == recipe_name, cols]
results.head(1)

,Recipe Name,is_vegetarian,is_vegan,allergens_present,is_dairy_free,is_gluten_free,is_keto,is_paleo,is_halal,is_kosher,is_hindu_friendly
1406668,Low-Carb Glazed Ribs,True,True,NaN,True,True,True,True,True,True,True


In [ ]:

spice_cols = list(mlb.classes_)
print('italian seasoning in mlb:', 'italian seasoning' in spice_cols)

if 'italian seasoning' in spice_cols:
    idx = spice_cols.index('italian seasoning')
    print(f'recipes with italian seasoning: {X[:, idx].sum():,}')
    print(f'that is {X[:, idx].mean()*100:.2f}% of recipes')

In [37]:

m = joblib.load("spicerack_model2.joblib")
print(list(m.keys()))
print("n_recipes:", m["n_recipes"])
print("n_clusters:", m["n_clusters"])

['kmeans', 'svd', 'mlb', 'tfidf', 'idf_boost', 'recipe_matrix', 'recipe_titles', 'recipe_spices', 'cluster_labels', 'cluster_top_spices', 'n_clusters', 'n_recipes', 'silhouette']
n_recipes: 2231142
n_clusters: 100


In [ ]:
test = pd.read_csv('cluster_data.csv')
test_copy = test.copy()
test_1000 = test_copy.sample(n=1000, random_state=42).reset_index(drop=True)



In [10]:
test.shape

(2231142, 10)

In [ ]:
test.columns

Index(['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
       'NER', 'ingredients_parsed', 'spices', 'cluster'],
      dtype='object')

In [14]:
test.head(10)

,Unnamed: 0,title,ingredients,directions,link,source,NER,ingredients_parsed,spices,cluster
0,0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu...","['brown sugar', 'milk', 'vanilla', 'nuts', 'bu...",{'vanilla'},118
1,1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom...","['beef', 'chicken breasts', 'cream of mushroom...",set(),108
2,2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar...","['frozen corn', 'cream cheese', 'butter', 'gar...","{'salt', 'garlic'}",97
3,3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroo...","['chicken', 'chicken gravy', 'cream of mushroo...",set(),108
4,4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""bu...","['peanut butter', 'graham cracker crumbs', 'bu...",set(),108
5,5,Cheeseburger Potato Soup,"[""6 baking potatoes"", ""1 lb. of extra lean gro...","[""Wash potatoes; prick several times with a fo...",www.cookbooks.com/Recipe-Details.aspx?id=20115,Gathered,"[""baking potatoes"", ""extra lean ground beef"", ...","['baking potatoes', 'extra lean ground beef', ...",{'salt'},111
6,6,Rhubarb Coffee Cake,"[""1 1/2 c. sugar"", ""1/2 c. butter"", ""1 egg"", ""...","[""Cream sugar and butter."", ""Add egg and beat ...",www.cookbooks.com/Recipe-Details.aspx?id=210288,Gathered,"[""sugar"", ""butter"", ""egg"", ""buttermilk"", ""flou...","['sugar', 'butter', 'egg', 'buttermilk', 'flou...","{'salt', 'vanilla'}",122
7,7,Scalloped Corn,"[""1 can cream-style corn"", ""1 can whole kernel...","[""Mix together both cans of corn, crackers, eg...",www.cookbooks.com/Recipe-Details.aspx?id=876969,Gathered,"[""cream-style corn"", ""whole kernel corn"", ""cra...","['cream-style corn', 'whole kernel corn', 'cra...",set(),108
8,8,Nolan'S Pepper Steak,"[""1 1/2 lb. round steak (1-inch thick), cut in...","[""Roll steak strips in flour."", ""Brown in skil...",www.cookbooks.com/Recipe-Details.aspx?id=375254,Gathered,"[""tomatoes"", ""water"", ""onions"", ""Worcestershir...","['tomatoes', 'water', 'onions', 'Worcestershir...",set(),108
9,9,Millionaire Pie,"[""1 large container Cool Whip"", ""1 large can c...","[""Empty Cool Whip into a bowl."", ""Drain juice ...",www.cookbooks.com/Recipe-Details.aspx?id=794547,Gathered,"[""pineapple"", ""condensed milk"", ""lemons"", ""pec...","['pineapple', 'condensed milk', 'lemons', 'pec...",set(),108
